<a href="https://colab.research.google.com/github/ebuseyy/SGuA-projects/blob/main/BeforeRevision/GWO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

# Define the number of relays and the number of wolves (population size)
num_relays = 6
num_wolves = 100  # You can adjust the population size

# Define the maximum and minimum values for parameters
ti_min = 1.0
ti_max = 2.2
td_min = 0.2
td_max = 1.0
Ikd1 = 1263
Ikd2_6 = 5639
Ip_min = [117.152, 523, 400, 500, 600, 400]
Ip_max = [128.65, 575, 420, 510, 610, 420]

def calculate_ti(td, Ikd, RIP):
    ti = [0] * num_relays
    for k in range(num_relays):
        x = (Ikd[k] / RIP[k]) ** 0.02
        calculated_ti = 0.14 * td[k] / (x - 1)
        # Soft constraint handling
        if calculated_ti < ti_min:
            ti[k] = ti_min + (ti_min - calculated_ti) * 0.1  # Penalize but don't set to boundary
        elif calculated_ti > ti_max:
            ti[k] = ti_max - (calculated_ti - ti_max) * 0.1  # Penalize but don't set to boundary
        else:
            ti[k] = calculated_ti
    return ti

# Define the objective function (fitness function)
def evaluate_fitness(td, Ikd, RIP):
    # Ensure td is within the specified range (Constraint 2)
    if any(t < td_min or t > td_max for t in td):
        return float('inf')  # Penalize invalid solutions

    # Ensure Ikd is within the specified range
    if any(Ikd_val < Ikd2_6 for Ikd_val in Ikd[1:]):
        return float('inf')  # Penalize invalid solutions

    # Ensure Ip is within the specified range (Constraint 3)
    for i in range(num_relays):
        if RIP[i] < Ip_min[i] or RIP[i] > Ip_max[i]:
            return float('inf')  # Penalize invalid solutions

    # Calculate relay operation times
    relay_times = calculate_ti(td, Ikd, RIP)

    # Check if relay_times are within the specified range (Constraint 1)
    for i in range(num_relays):
        if relay_times[i] < ti_min or relay_times[i] > ti_max:
            return float('inf')  # Penalize invalid solutions

    # Apply the second constraint
    # t1 should be greater than t2 by at least 0.3 seconds
    if relay_times[0] <= relay_times[1] + 0.3:
        return float('inf')

    # Apply the updated constraint: t2 should be greater than other relays (t3, t4, t5, t6) by at least 0.3 seconds
    min_other_relays = min(relay_times[2:])  # Find the minimum of t3, t4, t5, t6
    if relay_times[1] <= min_other_relays + 0.3:
        return float('inf')

    # Minimize the sum of relay operation times
    return sum(relay_times)

# Initialize wolf pack (population)
class Wolf:
    def __init__(self):
        self.td = np.random.uniform(td_min, td_max, size=num_relays)
        self.Ikd = [Ikd1] + [Ikd2_6] * (num_relays - 1)
        self.RIP = [np.random.uniform(low=Ip_min[i], high=Ip_max[i]) for i in range(num_relays)]
        self.fitness = evaluate_fitness(self.td, self.Ikd, self.RIP)

def run_gwo_algorithm(max_iterations=1000, a_initial=2):
    # Initialize wolf pack (population)
    population = [Wolf() for _ in range(num_wolves)]

    # Define GWO algorithm parameters
    a = a_initial  # Coefficient that decreases over iterations to help convergence

    # Main GWO loop
    for iteration in range(max_iterations):
        # Sort wolves by fitness to identify the alpha, beta, and delta wolves
        population = sorted(population, key=lambda wolf: wolf.fitness)
        alpha, beta, delta = population[:3]

        for wolf in population:
            new_td = wolf.td.copy()
            # For each wolf in the population, update its position
            for i in range(num_relays):
                # Calculate the coefficient components
                # These coefficients determine how far a wolf will move towards a leading wolf
                A1 = 2 * a * np.random.rand() - a
                C1 = 2 * np.random.rand()
                D_alpha = abs(C1 * alpha.td[i] - wolf.td[i])
                X1 = alpha.td[i] - A1 * D_alpha

                A2 = 2 * a * np.random.rand() - a
                C2 = 2 * np.random.rand()
                D_beta = abs(C2 * beta.td[i] - wolf.td[i])
                X2 = beta.td[i] - A2 * D_beta

                A3 = 2 * a * np.random.rand() - a
                C3 = 2 * np.random.rand()
                D_delta = abs(C3 * delta.td[i] - wolf.td[i])
                X3 = delta.td[i] - A3 * D_delta

                # Update the wolf's position by averaging the positions suggested by the alpha, beta, and delta wolves
                new_td[i] = (X1 + X2 + X3) / 3

            # Ensure the new position is within the specified bounds
            new_td = np.clip(new_td, td_min, td_max)
            # Recalculate fitness based on the new position
            new_fitness = evaluate_fitness(new_td, wolf.Ikd, wolf.RIP)
            # Update wolf's position and fitness if the new solution is better and feasible
            if new_fitness < wolf.fitness and new_fitness != float('inf'):
                wolf.td = new_td
                wolf.fitness = new_fitness

        # Reduce the value of 'a' over iterations for convergence from exploration to exploitation
        a *= (1 - iteration / max_iterations)

    # Identify the best solution after the final iteration
    best_wolf = min(population, key=lambda wolf: wolf.fitness)

    # Ensure the best solution found satisfies all constraints
    if best_wolf.fitness == float('inf'):
        print("No feasible solution found.")
        return None, None
    else:
        # Calculate relay operating times using the provided calculate_ti() function
        relay_ti = calculate_ti(best_wolf.td, best_wolf.Ikd, best_wolf.RIP)
        return best_wolf, relay_ti

# Call the GWO algorithm function
best_wolf, relay_ti = run_gwo_algorithm()

# Print the best solution and relay operating times
if best_wolf is not None:
    print("Table 1. GWO Results.")
    print("{:<10} {:<12} {:<10} {:<10}".format("Relay", "td", "Ip(A)", "Ti(s)"))
    for i in range(num_relays):
        print("{:<10} {:<12.6f} {:<10.3f} {:<10.3f}".format("Relay-{}".format(i + 1), best_wolf.td[i], best_wolf.RIP[i], relay_ti[i]))
    print("Best Fitness (minimized relay operation times):", best_wolf.fitness)


Table 1. GWO Results.
Relay      td           Ip(A)      Ti(s)     
Relay-1    0.548453     121.665    1.603     
Relay-2    0.447329     537.372    1.301     
Relay-3    0.307065     413.331    1.020     
Relay-4    0.333652     508.962    1.005     
Relay-5    0.325118     607.778    1.000     
Relay-6    0.201494     415.592    1.047     
Best Fitness (minimized relay operation times): 6.97605587868321
